# Module 2: Evaluation & Baseline Testing

## Overview

In this module, you'll establish a baseline for your multi-agent system using **two evaluation frameworks**:

1. **strands-evals (Online Evaluation)** - For quick interactive evaluation during development
2. **DeepEval (Offline Batch Evaluation)** - For comprehensive evaluation with experiment tracking

You will:
1. **Run evaluation with strands-evals** for quick feedback
2. **Run DeepEval batch evaluation** with versioned dataset
3. **Track experiments** with version control for reproducibility
4. **Check production readiness** against defined thresholds

### Evaluation Framework Comparison

| Feature | strands-evals | DeepEval |
|---------|---------------|----------|
| **Use Case** | Online eval, quick feedback | Offline batch eval, CI/CD |
| **Ground Truth** | Optional | Recommended |
| **Experiment Tracking** | No | Yes (via experiment_tracker) |
| **Production Stage** | Stage 2 (Production) | Stage 1 (Prototype) & Stage 3 (Improvement) |

### Learning Objectives
1. Understand evaluation strategies for multi-agent systems
2. Use strands-evals for quick interactive evaluation
3. Use DeepEval for comprehensive batch evaluation with ground truth
4. Track experiments with version control
5. Determine production readiness based on thresholds

### Time: ~90 minutes

## Step 1: Setup

Install dependencies and create the customer service instance.

In [1]:
# Install dependencies
!pip install -q strands-agents strands-agents-evals boto3 pandas deepeval


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


In [2]:
import json
import os
import sys
import boto3
import pandas as pd
from datetime import datetime

# Try to load REGION from Module 1
try:
    %store -r REGION
    print(f"✓ Loaded REGION from Module 1: {REGION}")
except:
    print("Could not load REGION from Module 1, using session default")
    session = boto3.Session()
    REGION = session.region_name or 'us-west-2'

print(f"Region: {REGION}")

# Set environment variable
os.environ['AWS_REGION'] = REGION

# Always create new customer_service instance (cannot be pickled)
print("\nCreating customer service instance...")
sys.path.insert(0, '../01-multi-agent-prototype/agents')
from orchestrator import MultiAgentCustomerService
customer_service = MultiAgentCustomerService(region=REGION)
print("✓ Customer service initialized")

✓ Loaded REGION from Module 1: us-west-2
Region: us-west-2

Creating customer service instance...
✓ Customer service initialized


## Step 2: Load Evaluation Experiment

Load the test cases and convert them to the format expected by strands-evals.

In [3]:
# Import strands-evals
from strands_evals import Experiment, Case
from strands_evals.evaluators import OutputEvaluator

# Load evaluation dataset
with open('evaluation_dataset.json', 'r') as f:
    eval_data = json.load(f)

print(f"Loaded {len(eval_data['test_cases'])} test cases")
print(f"\nSample test case:")
print(json.dumps(eval_data['test_cases'][0], indent=2))

Loaded 25 test cases

Sample test case:
{
  "id": "order_001",
  "category": "order_status",
  "difficulty": "easy",
  "input": "What's the status of order ORD-2024-10002?",
  "expected_agent": "order_agent",
  "expected_tools": [
    "get_order_status"
  ],
  "expected_output_contains": [
    "shipped",
    "UPS",
    "tracking"
  ],
  "ground_truth": "Order ORD-2024-10002 is shipped via UPS with tracking number 1Z999AA10123456784"
}


In [4]:
# Convert test cases to Case objects
test_cases = []
for tc in eval_data['test_cases']:
    case = Case(
        name=tc['id'],
        input=tc['input'],
        expected_output=tc.get('ground_truth', ''),
        metadata={
            'category': tc['category'],
            'expected_agent': tc.get('expected_agent'),
            'expected_tools': tc.get('expected_tools', []),
            'expected_output_contains': tc.get('expected_output_contains', [])
        }
    )
    test_cases.append(case)

print(f"Created {len(test_cases)} Case objects")
print(f"\nCategories: {set(tc.metadata['category'] for tc in test_cases)}")

Created 25 Case objects

Categories: {'out_of_scope', 'suspended_account', 'payment_methods', 'edge_case', 'address_update', 'order_return', 'product_inventory', 'order_not_found', 'order_history', 'product_policy', 'order_status', 'password_reset', 'order_tracking', 'order_cancellation', 'membership_benefits', 'refund_status', 'multi_agent', 'product_comparison', 'product_recommendation', 'product_details', 'product_search', 'account_info'}


## Step 3: Define Task Function

Create a function that runs the agent on each test case.

In [5]:
def run_customer_service_task(case: Case) -> str:
    """
    Task function that runs a test case through the multi-agent system.
    
    Args:
        case: Test case to run
        
    Returns:
        str: Agent response
    """
    try:
        response = customer_service.chat(case.input)
        return str(response)
    except Exception as e:
        return f"Error: {str(e)}"

print("Task function defined")
print("\nTesting task function with one case...")
test_response = run_customer_service_task(test_cases[0])
print(f"Response: {test_response[:200]}...")

Task function defined

Testing task function with one case...

Tool #1: delegate_to_order_agent

Tool #1: get_order_status
Great! Here's the status of your order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is on its way! You can track your shipment using the UPS tracking number provided above. Would you like me to get more detailed tracking information or help you with anything else?Great news! Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99

## Step 4: Run Evaluation with Built-in Evaluators

Use OutputEvaluator to evaluate goal success and helpfulness.

**Note:** Each Experiment can only have ONE evaluator, so we create separate experiments for each metric.

In [6]:
# Create Goal Success Evaluator
goal_success_evaluator = OutputEvaluator(
    rubric="""Evaluate if the agent response successfully addresses the customer's request.

Score 1.0: The response fully addresses the customer's request with accurate, helpful information
Score 0.75: The response mostly addresses the request but may be missing minor details
Score 0.5: The response partially addresses the request but has significant gaps
Score 0.25: The response attempts to address the request but fails to provide useful help
Score 0.0: The response does not address the request at all or provides incorrect information

Respond with a JSON object containing 'score' (float 0-1) and 'reasoning' (string)."""
)

print("Goal Success Evaluator created")

Goal Success Evaluator created


In [7]:
# Run Goal Success Evaluation (using subset for demo)
# Remove [:5] to run all test cases
goal_success_experiment = Experiment(
    cases=test_cases[:5],  # First 5 test cases for demo
    evaluators=[goal_success_evaluator]  # Pass as list
)

print("Running goal success evaluation...")
goal_success_results = goal_success_experiment.run_evaluations(run_customer_service_task)

Running goal success evaluation...

Tool #2: delegate_to_order_agent

Tool #2: get_order_status
Here's the status of your order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:

In [8]:


# Extract the report (first and only item in results list)
goal_success_report = goal_success_results[0]

print("\n" + "="*60)
print("GOAL SUCCESS EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], goal_success_report.scores, goal_success_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Goal Success Score: {goal_success_report.overall_score:.2f}")
print(f"Pass Rate: {sum(goal_success_report.test_passes)}/{len(goal_success_report.test_passes)}")
print(f"{'='*60}")


GOAL SUCCESS EVALUATION RESULTS

1. order_001 (order_status)
   Score: 1.00
   Reasoning: The output fully addresses the customer's order status inquiry. It provides all the key information from the expected output (order ORD-2024-10002 is ...

2. order_002 (order_tracking)
   Score: 1.00
   Reasoning: The output fully addresses the customer's shipment tracking request. It provides all the key information from the expected output (DHL carrier, tracki...

3. order_003 (order_return)
   Score: 0.00
   Reasoning: The output directly contradicts the expected outcome. The expected output indicates the return should be processed successfully for a delivered order ...

4. order_004 (order_cancellation)
   Score: 0.00
   Reasoning: The output provides completely incorrect information. The customer wants to cancel order ORD-2024-10005, and the expected output shows the order is pe...

5. order_005 (order_history)
   Score: 1.00
   Reasoning: The output fully addresses the customer's request by

In [9]:
# Create Helpfulness Evaluator
helpfulness_evaluator = OutputEvaluator(
    rubric="""Evaluate how helpful the agent response is for the customer.

Score 1.0: Extremely helpful - provides clear, actionable information and anticipates follow-up needs
Score 0.75: Very helpful - provides good information that addresses the customer's needs
Score 0.5: Somewhat helpful - provides basic information but could be more detailed
Score 0.25: Minimally helpful - provides limited useful information
Score 0.0: Not helpful - does not provide any useful information

Respond with a JSON object containing 'score' (float 0-1) and 'reasoning' (string)."""
)

# Run Helpfulness Evaluation
helpfulness_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[helpfulness_evaluator]  # Pass as list
)

print("Running helpfulness evaluation...")
helpfulness_results = helpfulness_experiment.run_evaluations(run_customer_service_task)

# Extract the report
helpfulness_report = helpfulness_results[0]

print("\n" + "="*60)
print("HELPFULNESS EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], helpfulness_report.scores, helpfulness_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Helpfulness Score: {helpfulness_report.overall_score:.2f}")
print(f"Pass Rate: {sum(helpfulness_report.test_passes)}/{len(helpfulness_report.test_passes)}")
print(f"{'='*60}")

Running helpfulness evaluation...

Tool #7: delegate_to_order_agent

Tool #7: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z9

## Step 5: Custom Evaluators

Run domain-specific custom evaluators for routing accuracy, policy compliance, and response quality.

In [10]:
  # Reload custom evaluators module (in case it was already imported)
  import importlib
  import sys
  if 'custom_evaluators' in sys.modules:
      import custom_evaluators
      importlib.reload(custom_evaluators)

In [11]:
# Import custom evaluators
from custom_evaluators import (
    RoutingAccuracyEvaluator,
    PolicyComplianceEvaluator,
    ResponseQualityEvaluator,
    CustomerSatisfactionEvaluator
)

print("Custom evaluators imported")

Custom evaluators imported


In [12]:
# Routing Accuracy Evaluation
routing_evaluator = RoutingAccuracyEvaluator()
routing_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[routing_evaluator]  # Pass as list
)

print("Running routing accuracy evaluation...")
routing_results = routing_experiment.run_evaluations(run_customer_service_task)

# Extract the report
routing_report = routing_results[0]

print("\n" + "="*60)
print("ROUTING ACCURACY EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], routing_report.scores, routing_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Routing Accuracy Score: {routing_report.overall_score:.2f}")
print(f"Pass Rate: {sum(routing_report.test_passes)}/{len(routing_report.test_passes)}")
print(f"{'='*60}")

Running routing accuracy evaluation...

Tool #12: delegate_to_order_agent

Tool #12: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number

In [15]:
# Policy Compliance Evaluation
policy_evaluator = PolicyComplianceEvaluator()
policy_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[policy_evaluator]  # Pass as list
)

print("Running policy compliance evaluation...")
policy_results = policy_experiment.run_evaluations(run_customer_service_task)

# Extract the report
policy_report = policy_results[0]

print("\n" + "="*60)
print("POLICY COMPLIANCE EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], policy_report.scores, policy_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Policy Compliance Score: {policy_report.overall_score:.2f}")
print(f"Pass Rate: {sum(policy_report.test_passes)}/{len(policy_report.test_passes)}")
print(f"{'='*60}")

Running policy compliance evaluation...
I'll check the status of order ORD-2024-10002 for you.
Tool #22: delegate_to_order_agent

Tool #22: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shippin

In [18]:
# Response Quality Evaluation
quality_evaluator = ResponseQualityEvaluator()
quality_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[quality_evaluator]  # Pass as list
)

print("Running response quality evaluation...")
quality_results = quality_experiment.run_evaluations(run_customer_service_task)

# Extract the report
quality_report = quality_results[0]

print("\n" + "="*60)
print("RESPONSE QUALITY EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], quality_report.scores, quality_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Response Quality Score: {quality_report.overall_score:.2f}")
print(f"Pass Rate: {sum(quality_report.test_passes)}/{len(quality_report.test_passes)}")
print(f"{'='*60}")

Running response quality evaluation...
I'll check the status of order ORD-2024-10002 for you.
Tool #27: delegate_to_order_agent

Tool #27: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping

In [19]:
# Customer Satisfaction Evaluation
satisfaction_evaluator = CustomerSatisfactionEvaluator()
satisfaction_experiment = Experiment(
    cases=test_cases[:5],
    evaluators=[satisfaction_evaluator]  # Pass as list
)

print("Running customer satisfaction evaluation...")
satisfaction_results = satisfaction_experiment.run_evaluations(run_customer_service_task)

# Extract the report
satisfaction_report = satisfaction_results[0]

print("\n" + "="*60)
print("CUSTOMER SATISFACTION EVALUATION RESULTS")
print("="*60)

# Display concise summary
for i, (case, score, reason) in enumerate(zip(test_cases[:5], satisfaction_report.scores, satisfaction_report.reasons), 1):
    print(f"\n{i}. {case.name} ({case.metadata['category']})")
    print(f"   Score: {score:.2f}")
    print(f"   Reasoning: {reason[:150]}..." if len(reason) > 150 else f"   Reasoning: {reason}")

print(f"\n{'='*60}")
print(f"Overall Customer Satisfaction Score: {satisfaction_report.overall_score:.2f}")
print(f"Pass Rate: {sum(satisfaction_report.test_passes)}/{len(satisfaction_report.test_passes)}")
print(f"{'='*60}")

Running customer satisfaction evaluation...
I'll check the status of order ORD-2024-10002 for you.
Tool #32: delegate_to_order_agent

Tool #32: get_order_status
Here's the status of order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your shipment using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shi

## Step 6: Extract and Analyze Results

Collect scores from all evaluations and compute baseline metrics.

In [20]:
# Helper function to extract scores from EvaluationReport
def extract_scores_from_report(report):
    """Extract scores from evaluation report"""
    return report.scores

# Extract all scores
goal_success_scores = extract_scores_from_report(goal_success_report)
helpfulness_scores = extract_scores_from_report(helpfulness_report)
routing_scores = extract_scores_from_report(routing_report)
policy_scores = extract_scores_from_report(policy_report)
quality_scores = extract_scores_from_report(quality_report)
satisfaction_scores = extract_scores_from_report(satisfaction_report)

print("✓ Scores extracted from all evaluations")
print(f"\nGoal Success: {goal_success_scores}")
print(f"Helpfulness: {helpfulness_scores}")
print(f"Routing: {routing_scores}")
print(f"Policy: {policy_scores}")
print(f"Quality: {quality_scores}")
print(f"Satisfaction: {satisfaction_scores}")

✓ Scores extracted from all evaluations

Goal Success: [1.0, 1.0, 0.0, 0.0, 1.0]
Helpfulness: [1.0, 1.0, 0.75, 1.0, 1.0]
Routing: [1.0, 1.0, 1.0, 1.0, 1.0]
Policy: [1.0, 1.0, 1.0, 0.0, 1.0]
Quality: [1.0, 1.0, 0.8, 0.2, 0.9]
Satisfaction: [1.0, 1.0, 0.25, 0.75, 1.0]


In [21]:
# Create DataFrame with all results
results_df = pd.DataFrame({
    'test_case': [case.name for case in test_cases[:5]],
    'category': [case.metadata['category'] for case in test_cases[:5]],
    'goal_success': goal_success_scores,
    'helpfulness': helpfulness_scores,
    'routing_accuracy': routing_scores,
    'policy_compliance': policy_scores,
    'response_quality': quality_scores,
    'customer_satisfaction': satisfaction_scores
})

print("\nEvaluation Results DataFrame:")
print(results_df.to_string(index=False))


Evaluation Results DataFrame:
test_case           category  goal_success  helpfulness  routing_accuracy  policy_compliance  response_quality  customer_satisfaction
order_001       order_status           1.0         1.00               1.0                1.0               1.0                   1.00
order_002     order_tracking           1.0         1.00               1.0                1.0               1.0                   1.00
order_003       order_return           0.0         0.75               1.0                1.0               0.8                   0.25
order_004 order_cancellation           0.0         1.00               1.0                0.0               0.2                   0.75
order_005      order_history           1.0         1.00               1.0                1.0               0.9                   1.00


## Step 7: Baseline Metrics

Calculate average scores to establish baseline performance.

In [22]:
# Calculate baseline metrics
baseline_metrics = {
    'timestamp': datetime.now().isoformat(),
    'total_test_cases': len(results_df),
    'goal_success_rate': results_df['goal_success'].mean(),
    'helpfulness': results_df['helpfulness'].mean(),
    'routing_accuracy': results_df['routing_accuracy'].mean(),
    'policy_compliance': results_df['policy_compliance'].mean(),
    'response_quality': results_df['response_quality'].mean(),
    'customer_satisfaction': results_df['customer_satisfaction'].mean()
}

print("\n" + "="*60)
print("BASELINE METRICS")
print("="*60)
for metric, value in baseline_metrics.items():
    if isinstance(value, float) and value <= 1.0:
        print(f"{metric:.<40} {value:.1%}")
    else:
        print(f"{metric:.<40} {value}")


BASELINE METRICS
timestamp............................... 2026-01-20T14:56:26.041788
total_test_cases........................ 5
goal_success_rate....................... 60.0%
helpfulness............................. 95.0%
routing_accuracy........................ 100.0%
policy_compliance....................... 80.0%
response_quality........................ 78.0%
customer_satisfaction................... 80.0%


In [23]:
# Performance by category
print("\n" + "="*60)
print("PERFORMANCE BY CATEGORY")
print("="*60)

category_metrics = results_df.groupby('category')[[
    'goal_success', 'helpfulness', 'routing_accuracy', 
    'policy_compliance', 'response_quality', 'customer_satisfaction'
]].mean()

print(category_metrics.to_string())


PERFORMANCE BY CATEGORY
                    goal_success  helpfulness  routing_accuracy  policy_compliance  response_quality  customer_satisfaction
category                                                                                                                   
order_cancellation           0.0         1.00               1.0                0.0               0.2                   0.75
order_history                1.0         1.00               1.0                1.0               0.9                   1.00
order_return                 0.0         0.75               1.0                1.0               0.8                   0.25
order_status                 1.0         1.00               1.0                1.0               1.0                   1.00
order_tracking               1.0         1.00               1.0                1.0               1.0                   1.00


## Step 8: Define Production Thresholds

Set alert thresholds based on baseline metrics (typically 80-90% of baseline).

In [24]:
# Define production thresholds
production_thresholds = {
    'goal_success_rate': 0.70,       # Alert if below 70%
    'helpfulness': 0.65,              # Alert if below 65%
    'routing_accuracy': 0.75,         # Alert if below 75%
    'policy_compliance': 0.80,        # Alert if below 80%
    'response_quality': 0.65,         # Alert if below 65%
    'customer_satisfaction': 0.70     # Alert if below 70%
}

print("\n" + "="*60)
print("PRODUCTION THRESHOLDS")
print("="*60)
for metric, threshold in production_thresholds.items():
    print(f"{metric:.<40} {threshold:.0%}")


PRODUCTION THRESHOLDS
goal_success_rate....................... 70%
helpfulness............................. 65%
routing_accuracy........................ 75%
policy_compliance....................... 80%
response_quality........................ 65%
customer_satisfaction................... 70%


In [25]:
# Check current performance against thresholds
print("\n" + "="*60)
print("THRESHOLD VALIDATION")
print("="*60)

all_pass = True
for metric, threshold in production_thresholds.items():
    current = baseline_metrics.get(metric, 0)
    passed = current >= threshold
    
    status = "✓ PASS" if passed else "✗ FAIL"
    
    if not passed:
        all_pass = False
    
    print(f"[{status}] {metric}: {current:.1%} (threshold: {threshold:.0%})")

print("\n" + "="*60)
if all_pass:
    print("✓ ALL THRESHOLDS PASSED - Ready for production!")
else:
    print("⚠ SOME THRESHOLDS FAILED - Review before production")
print("="*60)


THRESHOLD VALIDATION
[✗ FAIL] goal_success_rate: 60.0% (threshold: 70%)
[✓ PASS] helpfulness: 95.0% (threshold: 65%)
[✓ PASS] routing_accuracy: 100.0% (threshold: 75%)
[✓ PASS] policy_compliance: 80.0% (threshold: 80%)
[✓ PASS] response_quality: 78.0% (threshold: 65%)
[✓ PASS] customer_satisfaction: 80.0% (threshold: 70%)

⚠ SOME THRESHOLDS FAILED - Review before production


## Step 9: Save Results

Store evaluation results and baseline metrics for Module 3.

In [26]:
# Save detailed results
results_df.to_csv('evaluation_results.csv', index=False)
print("✓ Saved detailed results to evaluation_results.csv")

# Save baseline metrics
with open('baseline_metrics.json', 'w') as f:
    json.dump(baseline_metrics, f, indent=2, default=str)
print("✓ Saved baseline metrics to baseline_metrics.json")

# Store for next module
%store baseline_metrics
%store production_thresholds
%store REGION
print("\n✓ Data stored for Module 3!")

✓ Saved detailed results to evaluation_results.csv
✓ Saved baseline metrics to baseline_metrics.json
Stored 'baseline_metrics' (dict)
Stored 'production_thresholds' (dict)
Stored 'REGION' (str)

✓ Data stored for Module 3!


---

# Part 2: DeepEval Offline Batch Evaluation

The strands-evals above provides **quick feedback during development**. Now we'll run a comprehensive **offline batch evaluation** using **DeepEval** - the recommended approach for:

- **Stage 1 (Prototype)**: Validate agent before production deployment
- **Stage 3 (Continuous Improvement)**: Re-evaluate after making improvements

DeepEval provides:
- Ground truth comparison with expected behaviors
- Experiment tracking for reproducibility
- Production readiness validation against thresholds

## Step 10: Load Versioned Dataset

Load the evaluation dataset from the versioned datasets directory. This ensures reproducibility and tracks dataset changes over time.

In [6]:
# Load versioned dataset
from pathlib import Path

# Find the latest dataset version
datasets_dir = Path('../evaluation_datasets')
versions = sorted([d.name for d in datasets_dir.iterdir() if d.is_dir() and d.name.startswith('v')], reverse=True)
latest_version = versions[0] if versions else 'v1.0.0'

print(f"Available dataset versions: {versions}")
print(f"Using latest version: {latest_version}")

# Load the versioned dataset
dataset_path = datasets_dir / latest_version / 'dataset.json'
with open(dataset_path, 'r') as f:
    versioned_dataset = json.load(f)

print(f"\nDataset Version: {versioned_dataset['dataset_version']}")
print(f"Total Test Cases: {len(versioned_dataset['test_cases'])}")
print(f"Description: {versioned_dataset['description']}")

# Show new dataset structure with ground_truth
print("\nNew dataset structure (with ground_truth object):")
print(json.dumps(versioned_dataset['test_cases'][0], indent=2))

Available dataset versions: ['v1.0.0']
Using latest version: v1.0.0

Dataset Version: 1.0.0
Total Test Cases: 25
Description: Baseline evaluation dataset for E-Commerce Customer Service Multi-Agent System

New dataset structure (with ground_truth object):
{
  "id": "order_001",
  "category": "order_status",
  "difficulty": "easy",
  "input": "What's the status of order ORD-2024-10002?",
  "ground_truth": {
    "expected_agent": "order_agent",
    "expected_tools": [
      "get_order_status"
    ],
    "expected_response_contains": [
      "shipped",
      "UPS",
      "tracking"
    ],
    "expected_behavior": "Order ORD-2024-10002 is shipped via UPS with tracking number 1Z999AA10123456784"
  }
}


## Step 11: Setup DeepEval with Bedrock

Initialize DeepEval with our custom Bedrock model wrapper for evaluation.

In [11]:
# Import DeepEval components
from deepeval_config import (
    BedrockEvalModel, 
    get_deepeval_metrics, 
    create_test_case,
    PRODUCTION_THRESHOLDS,
    check_production_readiness
)
from deepeval import evaluate
from deepeval.test_case import LLMTestCase

# Initialize Bedrock model for evaluation
eval_model = BedrockEvalModel(
    model_id="global.anthropic.claude-sonnet-4-5-20250929-v1:0",
    region=REGION
)

print(f"DeepEval Model: {eval_model.get_model_name()}")
print(f"Region: {eval_model.region}")

# Get DeepEval metrics
deepeval_metrics = get_deepeval_metrics(eval_model)
print(f"\nAvailable DeepEval Metrics: {list(deepeval_metrics.keys())}")

# Show production thresholds
print("\nProduction Thresholds:")
for metric, threshold in PRODUCTION_THRESHOLDS.items():
    print(f"  {metric}: {threshold:.0%}")

DeepEval Model: global.anthropic.claude-sonnet-4-5-20250929-v1:0
Region: us-west-2

Available DeepEval Metrics: ['answer_relevancy', 'helpfulness', 'routing_accuracy', 'policy_compliance', 'tool_correctness']

Production Thresholds:
  answer_relevancy: 70%
  helpfulness: 65%
  routing_accuracy: 85%
  policy_compliance: 80%
  tool_correctness: 85%
  goal_success: 70%
  customer_satisfaction: 70%


## Step 12: Initialize Experiment Tracker

Create an experiment to track this evaluation run with version control.

In [8]:
# Import and initialize experiment tracker
from experiment_tracker import ExperimentTracker

# Create experiment tracker
tracker = ExperimentTracker('./experiments')

# Create a new experiment for this evaluation run
experiment_id = tracker.create_experiment(
    name="baseline_evaluation",
    tags={
        "stage": "prototype",
        "framework": "deepeval",
        "model": eval_model.get_model_name()
    },
    dataset_version=versioned_dataset['dataset_version'],
    hypothesis="Establish baseline metrics for production deployment"
)

print(f"Created Experiment: {experiment_id}")
print(f"Dataset Version: {versioned_dataset['dataset_version']}")
print(f"Experiments Directory: ./experiments/{experiment_id}/")

Created Experiment: exp_2026-01-20_16-38-26_baseline_evaluation
Dataset Version: 1.0.0
Experiments Directory: ./experiments/exp_2026-01-20_16-38-26_baseline_evaluation/


## Step 13: Run DeepEval Batch Evaluation

Run agent on all test cases and evaluate using DeepEval metrics with ground truth.

In [9]:
# Run agent on test cases and collect results
# Using first 5 cases for demo (remove [:5] to run all 25 cases)
NUM_CASES = 5  # Change to len(versioned_dataset['test_cases']) for full evaluation

print(f"Running DeepEval batch evaluation on {NUM_CASES} test cases...")
print("="*60)

deepeval_test_cases = []
agent_results = []

for i, tc in enumerate(versioned_dataset['test_cases'][:NUM_CASES]):
    case_id = tc['id']
    input_text = tc['input']
    ground_truth = tc.get('ground_truth', {})
    
    print(f"\n[{i+1}/{NUM_CASES}] {case_id}: {input_text[:50]}...")
    
    # Run agent
    try:
        agent_output = customer_service.chat(input_text)
        agent_output = str(agent_output)
    except Exception as e:
        agent_output = f"Error: {str(e)}"
    
    # Save agent result to experiment
    tracker.save_agent_result(
        experiment_id=experiment_id,
        case_id=case_id,
        input_text=input_text,
        output_text=agent_output,
        metadata={
            'category': tc.get('category'),
            'ground_truth': ground_truth
        }
    )
    
    # Create DeepEval test case with ground truth
    expected_output = ground_truth.get('expected_behavior', '')
    
    deepeval_case = LLMTestCase(
        input=input_text,
        actual_output=agent_output,
        expected_output=expected_output,
        context=[json.dumps(ground_truth)]  # Include full ground truth as context
    )
    deepeval_test_cases.append(deepeval_case)
    
    agent_results.append({
        'case_id': case_id,
        'category': tc.get('category'),
        'input': input_text,
        'output': agent_output[:200] + '...' if len(agent_output) > 200 else agent_output,
        'ground_truth': ground_truth
    })
    
    print(f"   Output: {agent_output[:100]}...")

print(f"\n{'='*60}")
print(f"Completed running {len(deepeval_test_cases)} test cases")

Running DeepEval batch evaluation on 5 test cases...

[1/5] order_001: What's the status of order ORD-2024-10002?...

Tool #2: delegate_to_order_agent

Tool #2: get_order_status
Here's the status of your order **ORD-2024-10002**:

**Order Status:** 🚚 **Shipped**

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order Total:** $359.97

**Shipping Information:**
- **Carrier:** UPS
- **Tracking Number:** 1Z999AA10123456784
- **Shipping To:** Los Angeles, CA 90001
- **Estimated Delivery:** January 8, 2025

Your order is currently in transit! You can track your package using the UPS tracking number. Is there anything else I can help you with?Your order **ORD-2024-10002** has been shipped! 🚚

**Order Status:** Shipped

**Order Details:**
- **Order Date:** December 28, 2024
- **Items:**
  - Smart Watch Pro (Qty: 1) - $299.99
  - Watch Band - Leather (Qty: 2) - $29.99 each
- **Order To

In [12]:
# Run DeepEval evaluation with selected metrics
from deepeval.evaluate import DisplayConfig

print("Running DeepEval metrics evaluation...")
print("="*60)

deepeval_results = {}

# Select metrics to run (using subset for demo)
selected_metrics = ['answer_relevancy', 'helpfulness']  # Add more as needed

for metric_name in selected_metrics:
    print(f"\nEvaluating: {metric_name}")
    metric = deepeval_metrics[metric_name]
    
    try:
        # Run evaluation with display config to suppress verbose output
        eval_result = evaluate(
            test_cases=deepeval_test_cases,
            metrics=[metric],
            display_config=DisplayConfig(print_results=False, show_indicator=False)
        )
        
        # Extract scores from test_results (correct way to access results)
        scores = []
        if hasattr(eval_result, 'test_results'):
            for test_result in eval_result.test_results:
                if test_result.metrics_data:
                    for metric_data in test_result.metrics_data:
                        if metric_data.score is not None:
                            scores.append(metric_data.score)
        
        avg_score = sum(scores) / len(scores) if scores else 0.0
        deepeval_results[metric_name] = {
            'scores': scores,
            'average': avg_score,
            'passed': avg_score >= PRODUCTION_THRESHOLDS.get(metric_name, 0.7)
        }
        
        print(f"  Scores: {[f'{s:.2f}' for s in scores]}")
        print(f"  Average: {avg_score:.2%}")
        print(f"  Threshold: {PRODUCTION_THRESHOLDS.get(metric_name, 0.7):.0%}")
        print(f"  Status: {'PASS' if deepeval_results[metric_name]['passed'] else 'FAIL'}")
        
    except Exception as e:
        print(f"  Error evaluating {metric_name}: {str(e)}")
        deepeval_results[metric_name] = {'scores': [], 'average': 0.0, 'passed': False, 'error': str(e)}

print(f"\n{'='*60}")
print("DeepEval evaluation complete")

Running DeepEval metrics evaluation...

Evaluating: answer_relevancy


⚠ WARNING: No hyperparameters logged.
» ]8;id=709595;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 48.8s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 100.0% | Passed: 5 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

  Scores: ['1.00', '1.00', '1.00', '1.00', '1.00']
  Average: 100.00%
  Threshold: 70%
  Status: PASS

Evaluating: helpfulness


⚠ WARNING: No hyperparameters logged.
» ]8;id=721428;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 40.37s | token cost: None)
» Test Results (5 total tests):
   » Pass Rate: 100.0% | Passed: 5 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

  Scores: ['1.00', '1.00', '0.80', '0.80', '1.00']
  Average: 92.00%
  Threshold: 65%
  Status: PASS

DeepEval evaluation complete


## Step 14: Save Results to Experiment Tracker

Save evaluation results to the experiment for version control and future comparison.

In [13]:
# Prepare results summary for experiment tracker
metrics_summary = {metric: data['average'] for metric, data in deepeval_results.items()}

# Prepare per-case results
per_case_results = []
for i, result in enumerate(agent_results):
    case_result = {
        'case_id': result['case_id'],
        'category': result['category'],
        'scores': {}
    }
    for metric_name, metric_data in deepeval_results.items():
        if i < len(metric_data['scores']):
            case_result['scores'][metric_name] = metric_data['scores'][i]
    per_case_results.append(case_result)

# Save to experiment tracker
tracker.save_eval_results(
    experiment_id=experiment_id,
    summary=metrics_summary,
    per_case=per_case_results,
    per_metric={metric: {'average': data['average'], 'passed': data['passed']} 
                for metric, data in deepeval_results.items()}
)

# Complete the experiment
tracker.complete_experiment(experiment_id)

print("Results saved to experiment tracker")
print(f"\nExperiment ID: {experiment_id}")
print(f"Results Location: ./experiments/{experiment_id}/")

# List experiment files
import os
exp_dir = f'./experiments/{experiment_id}'
if os.path.exists(exp_dir):
    print(f"\nExperiment Files:")
    for f in os.listdir(exp_dir):
        print(f"  - {f}")

Results saved to experiment tracker

Experiment ID: exp_2026-01-20_16-38-26_baseline_evaluation
Results Location: ./experiments/exp_2026-01-20_16-38-26_baseline_evaluation/

Experiment Files:
  - config.json
  - eval_results
  - agent_results
  - dataset_version.txt
  - tags.json


## Step 15: Production Readiness Check

Validate if the agent meets production thresholds for deployment.

In [14]:
# Check production readiness using DeepEval metrics
readiness_report = check_production_readiness(metrics_summary)

print("="*60)
print("PRODUCTION READINESS REPORT (DeepEval)")
print("="*60)

print(f"\nTimestamp: {readiness_report['timestamp']}")
print(f"\nMetrics Evaluated:")

for metric, data in readiness_report['metrics'].items():
    status = "PASS" if data['passed'] else "FAIL"
    score = data['score']
    threshold = data['threshold']
    print(f"  [{status}] {metric}: {score:.1%} (threshold: {threshold:.0%})")

print(f"\nPassed Metrics: {len(readiness_report['passed'])}")
print(f"Failed Metrics: {len(readiness_report['failed'])}")

print("\n" + "="*60)
if readiness_report['ready_for_production']:
    print("DECISION: READY FOR PRODUCTION")
    print("All evaluated metrics meet production thresholds.")
else:
    print("DECISION: NEEDS IMPROVEMENT")
    print(f"Failed metrics: {readiness_report['failed']}")
    print("\nRecommendation: Review failing cases and improve agent before deployment.")
print("="*60)

PRODUCTION READINESS REPORT (DeepEval)

Timestamp: 2026-01-20T16:49:50.786807

Metrics Evaluated:
  [PASS] answer_relevancy: 100.0% (threshold: 70%)
  [PASS] helpfulness: 92.0% (threshold: 65%)
  [FAIL] routing_accuracy: 0.0% (threshold: 85%)
  [FAIL] policy_compliance: 0.0% (threshold: 80%)
  [FAIL] tool_correctness: 0.0% (threshold: 85%)
  [FAIL] goal_success: 0.0% (threshold: 70%)
  [FAIL] customer_satisfaction: 0.0% (threshold: 70%)

Passed Metrics: 2
Failed Metrics: 5

DECISION: NEEDS IMPROVEMENT
Failed metrics: ['routing_accuracy', 'policy_compliance', 'tool_correctness', 'goal_success', 'customer_satisfaction']

Recommendation: Review failing cases and improve agent before deployment.


In [15]:
# Save DeepEval results
deepeval_baseline = {
    'timestamp': datetime.now().isoformat(),
    'experiment_id': experiment_id,
    'dataset_version': versioned_dataset['dataset_version'],
    'metrics': metrics_summary,
    'readiness_report': readiness_report,
    'num_cases': NUM_CASES
}

with open('deepeval_baseline.json', 'w') as f:
    json.dump(deepeval_baseline, f, indent=2, default=str)
    
print("Saved DeepEval baseline to deepeval_baseline.json")

# Store for Module 3
%store deepeval_baseline
%store experiment_id
print("\nDeepEval results stored for Module 3!")

Saved DeepEval baseline to deepeval_baseline.json
Stored 'deepeval_baseline' (dict)
Stored 'experiment_id' (str)

DeepEval results stored for Module 3!


---

## Module 2 Complete!

You have successfully:

1. **Part 1 (strands-evals)**: Ran quick interactive evaluation with custom evaluators
2. **Part 2 (DeepEval)**: Ran comprehensive batch evaluation with experiment tracking

### Files Created

| File | Description |
|------|-------------|
| `evaluation_results.csv` | strands-eval per-case results |
| `baseline_metrics.json` | strands-eval aggregate metrics |
| `deepeval_baseline.json` | DeepEval metrics and readiness report |
| `experiments/{id}/` | Full experiment with agent outputs, eval results |

### Next Steps

Based on your **Production Readiness Decision**:

- **READY FOR PRODUCTION** → Proceed to **Module 3: Production Deployment**
- **NEEDS IMPROVEMENT** → Review failing cases, improve agent, then re-run evaluation

### The Evaluation Lifecycle

```
Stage 1 (Prototype)     Stage 2 (Production)     Stage 3 (Improvement)
      │                        │                        │
      ▼                        ▼                        ▼
   DeepEval              strands-eval/             DeepEval
   Offline Eval         AgentCore Online          Re-evaluate
      │                        │                        │
      ▼                        ▼                        ▼
   Go/No-Go              Monitor & Alert         Validate Fix
```